# Notebook 5 - Run 3 LLM Enrichment

Clean build. Do not patch Notebook 4.

**Run 3 parameters:**
- Dataset: `data/sample_data3.parquet` filtered to `df_clean`
- Model: Claude 3.5 Sonnet v2 via AWS Bedrock
- Sample: ~500 rows stratified
- Output schema: 10 fields (incl. naive key, merchant, execution_path)
- Two parallel LLM calls per row: naive (6a) and full pipeline (6b)

**Sections:**
1. Setup and imports
2. Config (`run3_prompt_config.py`)
3. Data load and `df_clean` build
4. Context loading
5. Pre-processing pipeline
6. Prompt builder
7. Stratified sample
8. Batch loop with incremental save
9. Post-process and evaluation

---

## Changes from Notebook 4 (Run 2)

| Area | Notebook 4 | Notebook 5 |
|---|---|---|
| **Data source** | `df_sample_small.parquet` | `sample_data3.parquet` filtered to `df_clean` (X3 rows, SAVINGS, UNCATEGORISED, ROUND_UP mislabels removed) |
| **Sample size** | 500 | 1,000 |
| **Field name** | `dt_hint` | `spend_type` |
| **Biller lookup** | Returns category hint, continues to LLM | HIT short-circuits entire pipeline - no LLM call |
| **MCC lookup** | Returns category hint only | Returns `base_category_key` hint + sets `merchant_present_proxy` |
| **Merchant presence** | Not an explicit step | Explicit Step 4 - runs before NS regex engine. DynamoDB/Cocoflavor lookup deferred to Run 4 - LLM extraction used as baseline |
| **NS regex engine** | Runs on all rows | Runs on non-spend path only |
| **LLM calls per row** | 1 | 2 in parallel - naive (6a, no hints) + full pipeline (6b) |
| **Output schema** | 7 fields | 10 fields - adds `base_category_key_naive`, `merchant`, `execution_path` |
| **Confidence** | String: high / medium / low | Integer 1-10, calibrated from steps fired |
| **Reason** | Prose sentence | Keyword chain e.g. `mcc:5411->merchant:Woolworths->GROCERIES` |
| **Hint framing** | Follow signals | Evidence not hard rules - LLM may override if description conflicts |
| **Hint labels** | None | Importance tiers: `[critical]`, `[high]`, `[medium]` |
| **L1 GT resolution** | Raw `spend_type` from taxonomy (broken for mixed keys) | Resolved via `leans` / `manual_spend_override` from taxonomy master |
| **Post-process** | Flag / PO report | Coverage report (biller/mcc/dt/merchant/regex/llm-only%) + naive vs pipeline lift |
| **Config file** | `run2_prompt_config.py` | `run3_prompt_config.py` |

---
## 1. Setup and Imports

In [ ]:
import os
import re
import json
import time
import asyncio
import numpy as np
import pandas as pd
import boto3
from pathlib import Path
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

print('Imports OK')

In [ ]:
import threading

def keep_alive(interval=300):
    while True:
        threading.Event().wait(interval)
        _ = 1 + 1  # no-op

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print('Keep-alive thread started')

---
## 2. Config

All tunable parameters live in `run3_prompt_config.py`. Import once here; nothing hardcoded in notebook cells.

In [ ]:
from run3_prompt_config import (
    MODEL_ID, MAX_TOKENS, TEMPERATURE, BUDGET_USD, TARGET_ROWS,
    BASE_LAYER_ROWS, TOP_UP_ROWS, TOP_UP_PER_CAT, FOCUS_CATEGORIES,
    AGGREGATOR_DT_VALUE, MCC_SKIP, MCC_NON_SPEND, BILLER_CODE_ZEROS,
    PROVIDER_ANZ, PROVIDER_ING, PROVIDER_BANKWEST, PROVIDER_UP, PROVIDER_MACQUARIE,
    INVALID_KEY_BLOCKLIST, RULES_FILE, RULES_STRIP_PATTERN,
    OUTPUT_DIR, OUTPUT_FILE,
)

OUTPUT_DIR.mkdir(exist_ok=True)
print('Config loaded')
print(f'Output -> {OUTPUT_FILE}')

---
## 3. Data Load and df_clean Build

In [ ]:
df_sample = pd.read_parquet('data/sample_data3.parquet')
print(f'df_sample: {len(df_sample):,} rows, {df_sample.shape[1]} cols')
print(df_sample.dtypes)
df_sample.head(3)

In [ ]:
# Step 1 - remove X3 post-processed rows
df_sample['x3_processed'] = df_sample['properties'].str.contains(
    'ml_transfer_sub_category_key', case=False, na=False
)
df_work = df_sample[~df_sample['x3_processed']].copy()
print(f'After X3 filter: {len(df_work):,} rows (removed {len(df_sample) - len(df_work):,})')

# Step 2 - remove GT cleanup categories
mask_remove = (
    (df_work['base_category_key'] == 'SAVINGS') |
    (df_work['base_category_key'] == 'UNCATEGORISED') |
    (
        (df_work['base_category_key'] == 'INTERNAL_TRANSFER') &
        df_work['original_description'].str.contains(r'(?i)round.?up', na=False)
    )
)
df_clean = df_work[~mask_remove].copy()
print(f'After GT cleanup: {len(df_clean):,} rows (removed {mask_remove.sum():,})')

# Sanity check
print('\nGT distribution (top 20):')
print(df_clean['base_category_key'].value_counts().head(20))

---
## 4. Context Loading

Loads all reference files once. Expensive to re-read; keep in memory for batch loop.

In [ ]:
# Valid key gate
cat_keys_df  = pd.read_csv('assets/cat_keys.csv')
VALID_KEYS   = set(cat_keys_df['base_category_key'].dropna().str.strip())
print(f'Valid keys: {len(VALID_KEYS)}')

# Biller map - biller_code -> base_category_key
biller_df  = pd.read_csv('assets/biller_mapping.csv')
biller_map = dict(zip(
    biller_df.iloc[:, 0].astype(str).str.strip(),
    biller_df.iloc[:, 1].astype(str).str.strip()
))
print(f'Biller map: {len(biller_map)} entries')

# MCC map - mcc_code -> base_category_key
mcc_df  = pd.read_csv('assets/mcc_mapping.csv')
mcc_map = dict(zip(
    mcc_df.iloc[:, 0].astype(str).str.strip(),
    mcc_df.iloc[:, 1].astype(str).str.strip()
))
print(f'MCC map: {len(mcc_map)} entries')

# NS regex rules (layered_ns_rules + new_bank_patterns combined)
ns_rules_df = pd.read_csv('assets/layered_ns_rules.csv')
nbp_df      = pd.read_csv('assets/new_bank_patterns.csv')
ns_rules    = pd.concat([ns_rules_df, nbp_df], ignore_index=True)
print(f'NS regex rules: {len(ns_rules)} rows')

# Taxonomy master - for GT resolution and L2 eval
tax_df = pd.read_csv('assets/taxonomy_master_updated_override.csv') ##
print(f'Taxonomy master: {len(tax_df)} rows, cols: {list(tax_df.columns)}')

# Rules file - Sections 1+2 only, strip Section 3
rules_raw = Path(RULES_FILE).read_text()
# Strip everything from Section 3 onwards
sec3_match = re.search(RULES_STRIP_PATTERN, rules_raw, re.IGNORECASE | re.MULTILINE)
RULES_CONTEXT = rules_raw[:sec3_match.start()].strip() if sec3_match else rules_raw
print(f'Rules context: {len(RULES_CONTEXT):,} chars ({len(RULES_CONTEXT)//4:,} est. tokens)')

# Taxonomy context for prompt injection (grouped summary)
# Build from tax_df: spend_type -> category_group -> keys  /  non_spend: category_type -> keys
def build_taxonomy_context(df: pd.DataFrame) -> str:
    lines = ['VALID BASE_CATEGORY_KEY VALUES\n']
    for stype in ['spend', 'non_spend']:
        subset = df[df['spend_type'] == stype] if 'spend_type' in df.columns else df
        lines.append(f'--- {stype.upper()} ---')
        group_col = 'category_group' if stype == 'spend' else 'category_type'
        if group_col in df.columns:
            for grp, grp_df in subset.groupby(group_col, dropna=False):
                keys = ', '.join(sorted(grp_df['base_category_key'].dropna().unique()))
                lines.append(f'  {grp}: {keys}')
        else:
            keys = ', '.join(sorted(subset['base_category_key'].dropna().unique()))
            lines.append(f'  {keys}')
        lines.append('')
    return '\n'.join(lines)

tax_df['spend_type'] = tax_df.apply(
    lambda r: r['manual_spend_override']
    if pd.notna(r['manual_spend_override']) and r['manual_spend_override'] != ''
    else r['spend_type'],
    axis=1
)

TAXONOMY_CONTEXT = build_taxonomy_context(tax_df)
print(f'Taxonomy context: {len(TAXONOMY_CONTEXT):,} chars')

# GT spend_type resolver - resolves 'mixed' via leans / manual_spend_override
def build_gt_spend_map(df: pd.DataFrame) -> dict:
    """Returns {base_category_key: resolved_spend_type} using leans/manual_spend_override."""
    result = {}
    for _, row in df.iterrows():
        key = str(row.get('base_category_key', '')).strip()
        st  = str(row.get('spend_type', '')).strip()
        if st == 'mixed':
            override = str(row.get('manual_spend_override', '')).strip()
            leans    = str(row.get('leans', '')).strip()
            st = override if override and override != 'nan' else leans
        result[key] = st if st and st != 'nan' else None
    return result

GT_SPEND_MAP = build_gt_spend_map(tax_df)
print(f'GT spend map: {len(GT_SPEND_MAP)} keys')

---
## 5. Pre-Processing Pipeline

Functions for Steps 1-5. Called per-row in the batch loop.

In [ ]:
def _str(val) -> str:
    """Safe string coerce - treats None/nan as empty string."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return ''
    return str(val).strip()


# ---- Step 1: Biller lookup -----------------------------------------------

def resolve_biller(row: dict) -> Optional[str]:
    """Exact match on biller_code -> base_category_key. Returns None on miss."""
    bc = _str(row.get('biller_code', ''))
    if bc in BILLER_CODE_ZEROS:
        return None
    return biller_map.get(bc)


# ---- Step 2: MCC lookup --------------------------------------------------

def resolve_mcc(row: dict) -> tuple[Optional[str], bool]:
    """
    Match mcc -> base_category_key hint.
    Returns (category_key_hint, merchant_present_proxy).
    Excludes non-definitive MCC codes.
    """
    mcc = _str(row.get('merchant_category_code', ''))
    if mcc in MCC_SKIP:
        return None, False
    hit = mcc_map.get(mcc)
    return hit, (hit is not None)  # proxy: if MCC maps to a key, merchant-type txn


# ---- Step 3: Decision Tree -----------------------------------------------

def run_decision_tree(row: dict) -> tuple[str, Optional[str]]:
    """
    Python DT mirroring production BC logic.
    Only applies when aggregator == AGGREGATOR_DT_VALUE.
    Returns (spend_type_hint, rule_label).
    hint in {spend, non_spend, ambiguous, none}.
    """
    agg = _str(row.get('aggregator_id', row.get('aggregator', '')))
    if agg != AGGREGATOR_DT_VALUE:
        return 'none', None

    tt     = _str(row.get('transaction_type', ''))
    mcc    = _str(row.get('merchant_category_code', ''))
    bc     = _str(row.get('biller_code', ''))
    bn     = _str(row.get('biller_name', ''))
    mn     = _str(row.get('merchant_name', ''))
    prov   = _str(row.get('provider_name', '')).upper()
    try:
        amount = float(row.get('amount', 0) or 0)
    except (ValueError, TypeError):
        amount = 0.0

    bc_zero = bc in BILLER_CODE_ZEROS
    mcc_ok  = mcc and mcc not in MCC_SKIP

    if tt in {'0', '1', '2'}:                              return 'non_spend', 'Rule 1'
    if bc and not bc_zero:                                 return 'spend',     'Rule 2'
    if bn:                                                 return 'spend',     'Rule 3'
    if mcc in MCC_NON_SPEND:                               return 'non_spend', 'Rule 4'
    if mcc and amount == 0.0:                              return 'non_spend', 'Rule 5'
    if mcc_ok and PROVIDER_MACQUARIE not in prov:          return 'spend',     'Rule 6'
    if PROVIDER_ANZ in prov and mn:                        return 'spend',     'Rule 9'
    if PROVIDER_ANZ in prov and bc_zero:                   return 'non_spend', 'Rule 10'
    if PROVIDER_ING in prov and mn:                        return 'spend',     'Rule 11'
    if PROVIDER_BANKWEST in prov and mn:                   return 'spend',     'Rule 12'
    if PROVIDER_UP in prov and mn:                         return 'spend',     'Rule 13'
    return 'ambiguous', None


# ---- Step 4: Merchant presence -------------------------------------------

# Non-spend description tokens that look like merchants but aren't
_NS_TOKENS = re.compile(
    r'\b(ATO|CENTRELINK|MEDICARE|MYOB|BPAY|DIRECT DEBIT|PAYROLL|SALARY|'
    r'REFUND|REBATE|INTEREST|DIVIDEND|TRANSFER|PAYMENT FROM|PAYMENT TO|'
    r'DIRECT CREDIT|GOVERNMENT|PENSION|BENEFITS?|SUPERANNUATION)\b',
    re.IGNORECASE
)

def detect_merchant_presence(row: dict, mcc_proxy: bool = False) -> bool:
    """
    Step 4 - explicit merchant presence check.
    Primary: merchant_name field populated.
    Secondary: description tokens suggest a retail/merchant entity.
    Tertiary: MCC proxy flag from Step 2.
    Key challenge: avoid false positives from non-spend descriptions.
    """
    mn   = _str(row.get('merchant_name', ''))
    desc = _str(row.get('original_description', ''))

    # Primary: merchant_name populated and not a known non-spend pattern
    if mn and not _NS_TOKENS.search(mn):
        return True

    # Tertiary: MCC proxy
    if mcc_proxy:
        return True

    # Secondary: description looks like a retail entity (not a non-spend phrase)
    if desc and not _NS_TOKENS.search(desc):
        # Heuristic: short token count + mixed case or all-caps brand name
        tokens = desc.split()
        if 1 <= len(tokens) <= 6:
            return True

    return False


# ---- Step 5: NS regex engine (non-spend path only) -----------------------

def run_ns_regex(row: dict) -> Optional[str]:
    """
    Non-spend regex engine replicating M1func logic.
    Runs on layered_ns_rules + new_bank_patterns combined.
    Returns category_type string or None.
    Only call on non-spend path.
    """
    desc = _str(row.get('original_description', '')).upper()
    prov = _str(row.get('provider_name', '')).upper()
    for _, rule in ns_rules.iterrows():
        pattern  = _str(rule.get('pattern', ''))
        cat      = _str(rule.get('base_category_key', rule.get('category_type', ''))).upper()
        provider = _str(rule.get('provider', ''))
        if not pattern or not cat:
            continue
        # Provider-scoped rules: only apply if provider matches (or rule has no provider)
        if provider and provider.upper() not in prov:
            continue
        try:
            if re.search(pattern, desc, re.IGNORECASE):
                return cat
        except re.error:
            continue
    return None


# ---- Orchestrator: run all steps, return signals dict --------------------

def run_pipeline(row: dict) -> dict:
    """
    Runs Steps 1-5 for a single row.
    Returns signals dict consumed by prompt builder and batch loop.
    Biller HIT short-circuits: returns biller_hit=True + base_category_key.
    """
    # Step 1 - biller
    biller_key = resolve_biller(row)
    if biller_key:
        return {
            'biller_hit':         True,
            'biller_category_key': biller_key,
            'mcc_category_key':   None,
            'merchant_present_proxy': False,
            'spend_type_hint':    'spend',
            'dt_rule':            None,
            'merchant_present':   True,
            'regex_category_type': None,
            'execution_path_steps': ['biller'],
        }

    # Step 2 - MCC
    mcc_key, mcc_proxy = resolve_mcc(row)

    # Steps 3 + 4 - DT and merchant presence (conceptually parallel)
    spend_hint, dt_rule = run_decision_tree(row)
    merchant_present    = detect_merchant_presence(row, mcc_proxy=mcc_proxy)

    # Step 5 - NS regex (non-spend path only)
    is_non_spend = (spend_hint == 'non_spend') or (not merchant_present and spend_hint in {'none', 'ambiguous'})
    regex_cat    = run_ns_regex(row) if is_non_spend else None

    # Build execution_path_steps (only fired steps)
    steps = []
    if mcc_key:          steps.append('mcc')
    if spend_hint != 'none': steps.append('dt')
    if merchant_present: steps.append('merchant')
    if regex_cat:        steps.append('regex')

    return {
        'biller_hit':          False,
        'biller_category_key': None,
        'mcc_category_key':    mcc_key,
        'merchant_present_proxy': mcc_proxy,
        'spend_type_hint':     spend_hint,
        'dt_rule':             dt_rule,
        'merchant_present':    merchant_present,
        'regex_category_type': regex_cat,
        'execution_path_steps': steps,
    }


print('Pre-processing functions ready')

---
## 6. Prompt Builder

System prompt built once. Two user prompt builders: naive (6a) and full pipeline (6b).

In [ ]:
# ---- System prompt (built once) ------------------------------------------

def build_system_prompt() -> str:
    blocklist_str = ', '.join(INVALID_KEY_BLOCKLIST)
    return f"""You are a financial transaction categorisation engine.
Your task is to assign a single base_category_key from the taxonomy below to each transaction.

## OUTPUT FORMAT
Return a single JSON object with exactly these 9 fields (base_category_key_naive is omitted - provided separately):
{{
  "spend_type": "spend | non_spend",
  "category_group": "string (spend only, else null)",
  "category_type": "string (non_spend only, else null)",
  "base_category_key": "EXACT_KEY_FROM_TAXONOMY",
  "merchant": "Title Case name | null",
  "execution_path": "step->step->llm",
  "confidence": 7,
  "reason": "keyword:value->keyword:value->KEY",
  "flag": null
}}
Return ONLY the JSON object. No explanation, no reasoning, no text before or after.

## CONFIDENCE SCALE (integer 1-10)
Derive confidence from which pipeline steps fired, not just output certainty:
- 9-10: biller lookup hit (deterministic)
- 7-8: MCC lookup hit (specific MCC=8, broad/generic MCC=7) or strong DT signal
- 5-6: merchant presence + reasonable inference; or DT ambiguous (max 6)
- 3-4: LLM intuition only, no upstream hints fired

## REASON FORMAT
Keyword chain, not prose. Examples:
- "mcc:5411->merchant:Woolworths->GROCERIES"
- "dt:non_spend->regex:TRANSFERS->INTERNAL_TRANSFER"
- "no_hints->description:netflix.com/au->SUBSCRIPTIONS"

## EXECUTION PATH
Record only steps that fired, using -> separator. Include 'llm' at end.
Examples: "biller->llm", "mcc->dt->merchant->llm", "dt->regex->llm", "llm"

## MERCHANT FIELD
Extract and normalise merchant name from original_description.
- Title case, no location suffixes, no legal entity suffixes
- e.g. "Woolworths" not "WOOLWORTHS METRO SYDNEY PTY LTD"
- Return null if not identifiable (transfers, fees, government payments etc.)

## HINT PRIORITY ORDER
Hints are evidence, not hard rules. You may override if the description clearly conflicts.
1. biller_category_key [critical] - deterministic lookup; follow unless description is clearly wrong
2. mcc_category_key [high] - strong signal; override if merchant name contradicts
3. spend_type from DT [high] - follow unless other signals strongly disagree
4. merchant_present [medium] - lean signal for spend/non-spend split
5. regex_category_type [medium] - non-spend path only; confirms category_type
6. Your own merchant/brand/description knowledge [low-medium] - apply always

## REASONING PATHS
SPEND PATH: merchant name or MCC hint (strongest) -> description language -> amount as tiebreaker -> category_group -> base_category_key
NON-SPEND PATH: regex_category_type hint -> category_type -> base_category_key
INTEREST: amount > 0 -> INTEREST_INCOME; amount < 0 -> INTEREST_CHARGED

## INVALID KEYS - NEVER RETURN THESE
{blocklist_str}

## RULES
{RULES_CONTEXT}

## TAXONOMY
{TAXONOMY_CONTEXT}

## FLAG FIELD
Optional. Set to null unless you need to flag: taxonomy_gap, rule_conflict, ambiguous_boundary, ground_truth_suspicion.
Format: {{"type": "...", "detail": "one sentence"}}
"""


SYSTEM_PROMPT = build_system_prompt()
print(f'System prompt: {len(SYSTEM_PROMPT):,} chars (~{len(SYSTEM_PROMPT)//4:,} tokens)')

In [ ]:
# ---- User prompt: 6b full pipeline ---------------------------------------

def build_user_prompt(row: dict, signals: dict) -> str:
    """
    Builds the per-transaction user prompt for the full pipeline call (6b).
    Injects all fired signals with importance tier labels.
    """
    # Transaction fields
    provider  = _str(row.get('provider_name', ''))
    desc      = _str(row.get('original_description', ''))
    merch     = _str(row.get('merchant_name', ''))
    amount    = row.get('amount', '')
    tt        = _str(row.get('transaction_type', ''))
    at        = _str(row.get('account_type', ''))
    mcc       = _str(row.get('merchant_category_code', ''))
    bc        = _str(row.get('biller_code', ''))
    bn        = _str(row.get('biller_name', ''))

    # Signals with tier labels
    def _hint(val, label, tier):
        return f'{label}: {val} [{tier}]' if val else f'{label}: none'

    biller_line  = _hint(signals.get('biller_category_key'), 'biller_category_key', 'critical')
    mcc_line     = _hint(signals.get('mcc_category_key'),    'mcc_category_key',    'high')
    dt_line      = f"spend_type_hint: {signals.get('spend_type_hint', 'none')} [high]" + \
                   (f" ({signals['dt_rule']})" if signals.get('dt_rule') else '')
    merch_line   = f"merchant_present: {signals.get('merchant_present', False)} [medium]"
    regex_line   = _hint(signals.get('regex_category_type'), 'regex_category_type', 'medium')

    return f"""Transaction:
  provider:               {provider}
  description:            {desc}
  merchant_name:          {merch}
  amount:                 {amount}
  transaction_type:       {tt}
  account_type:           {at}
  merchant_category_code: {mcc}
  biller_code:            {bc}
  biller_name:            {bn}

Pipeline signals:
  {biller_line}
  {mcc_line}
  {dt_line}
  {merch_line}
  {regex_line}
"""


# ---- User prompt: 6a naive -----------------------------------------------

NAIVE_SYSTEM_PROMPT = f"""You are a financial transaction categorisation engine.
Given only a transaction description and amount, return the single most likely base_category_key.
Return ONLY a JSON object: {{"base_category_key_naive": "KEY"}}
No explanation, no reasoning, no text before or after.

## INVALID KEYS - NEVER RETURN THESE
{', '.join(INVALID_KEY_BLOCKLIST)}

## VALID KEYS
{TAXONOMY_CONTEXT}
"""

def build_naive_user_prompt(row: dict) -> str:
    desc   = _str(row.get('original_description', ''))
    amount = row.get('amount', '')
    return f"description: {desc}\namount: {amount}"


print('Prompt builders ready')

# Spot-check prompt on first row
test_row     = df_clean.iloc[0].to_dict()
test_signals = run_pipeline(test_row)
test_prompt  = build_user_prompt(test_row, test_signals)
print('\n--- Sample user prompt (6b) ---')
print(test_prompt)
print('\n--- Sample naive prompt (6a) ---')
print(build_naive_user_prompt(test_row))

---
## 7. Stratified Sample

In [ ]:
rng = np.random.default_rng(42)

# Base layer: distributed across providers
providers    = df_clean['provider_name'].dropna().unique()
rows_per_prov = max(1, BASE_LAYER_ROWS // len(providers))
base_frames  = []
for prov in providers:
    prov_df = df_clean[df_clean['provider_name'] == prov]
    n       = min(rows_per_prov, len(prov_df))
    base_frames.append(prov_df.sample(n=n, random_state=42))
df_base = pd.concat(base_frames).drop_duplicates()
print(f'Base layer: {len(df_base)} rows across {len(providers)} providers')

# Top-up layer: focus categories
topup_frames = []
for cat in FOCUS_CATEGORIES:
    cat_df = df_clean[
        (df_clean['base_category_key'] == cat) &
        (~df_clean.index.isin(df_base.index))
    ]
    n = min(TOP_UP_PER_CAT, len(cat_df))
    if n > 0:
        topup_frames.append(cat_df.sample(n=n, random_state=42))
df_topup = pd.concat(topup_frames).drop_duplicates() if topup_frames else pd.DataFrame()
print(f'Top-up layer: {len(df_topup)} rows from {len(topup_frames)} focus categories')

df_sample_run = pd.concat([df_base, df_topup]).drop_duplicates().reset_index(drop=True)
print(f'\nFinal sample: {len(df_sample_run)} rows')
print('GT distribution:')
print(df_sample_run['base_category_key'].value_counts().head(20))

---
## 8. Batch Loop with Incremental Save

Two parallel Bedrock calls per row (6a naive + 6b full pipeline).
Biller hits short-circuit: no LLM call needed.
Incremental save after every row.

In [ ]:
# ---- Bedrock client setup ------------------------------------------------

bedrock = boto3.client('bedrock-runtime', region_name='ap-southeast-2')


def call_bedrock(system_prompt: str, user_prompt: str, max_tokens: int = MAX_TOKENS, max_retries: int = 4) -> str:
    bedrock = boto3.client('bedrock-runtime', region_name='ap-southeast-2')
    body = json.dumps({
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': max_tokens,
        'temperature': TEMPERATURE,
        'system': system_prompt,
        'messages': [{'role': 'user', 'content': user_prompt}],
    })
    for attempt in range(max_retries):
        try:
            resp = bedrock.invoke_model(
                modelId=MODEL_ID,
                contentType='application/json',
                accept='application/json',
                body=body,
            )
            raw = json.loads(resp['body'].read())
            return raw['content'][0]['text']
        except Exception as e:
            if 'ThrottlingException' in str(type(e)) and attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise


def parse_json_response(text: str) -> dict:
    """Extract first JSON object from LLM response. Robust to trailing text."""
    cleaned = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*```$', '', cleaned.strip())
    match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if match:
        return json.loads(match.group())
    return json.loads(cleaned)
    

def process_row(idx: int, row: dict) -> dict:
    """
    Full pipeline for one transaction.
    Returns result dict with all 10 output fields + metadata.
    """
    result = {
        'row_idx':             idx,
        'original_description': row.get('original_description', ''),
        'provider_name':       row.get('provider_name', ''),
        'amount':              row.get('amount', ''),
        'gt_base_category_key': row.get('base_category_key', ''),
        # Pipeline output fields
        'spend_type':          None,
        'category_group':      None,
        'category_type':       None,
        'base_category_key':   None,
        'base_category_key_naive': None,
        'merchant':            None,
        'execution_path':      None,
        'confidence':          None,
        'reason':              None,
        'flag':                None,
        'parse_error':         False,
        'invalid_key':         False,
        'raw_response':        None,
        'raw_response_naive':  None,
    }

    signals = run_pipeline(row)

    # Biller short-circuit: no LLM needed
    if signals['biller_hit']:
        key = signals['biller_category_key']
        result.update({
            'base_category_key': key,
            'base_category_key_naive': key,  # same for naive - no point calling
            'spend_type':        'spend',
            'execution_path':    'biller->llm',
            'confidence':        10,
            'reason':            f'biller:{key}',
        })
        return result

    # Build prompts
    user_prompt_full  = build_user_prompt(row, signals)
    user_prompt_naive = build_naive_user_prompt(row)

    # Parallel calls: 6a naive + 6b full pipeline
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut_full  = ex.submit(call_bedrock, SYSTEM_PROMPT,       user_prompt_full,  MAX_TOKENS)
        fut_naive = ex.submit(call_bedrock, NAIVE_SYSTEM_PROMPT, user_prompt_naive, 100)
        raw_full  = fut_full.result()
        raw_naive = fut_naive.result()

    result['raw_response']       = raw_full
    result['raw_response_naive'] = raw_naive

    # Parse naive response
    try:
        naive_parsed = parse_json_response(raw_naive)
        result['base_category_key_naive'] = naive_parsed.get('base_category_key_naive')
    except Exception:
        result['base_category_key_naive'] = None

    # Parse full pipeline response
    try:
        parsed = parse_json_response(raw_full)
        result.update({
            'spend_type':     parsed.get('spend_type'),
            'category_group': parsed.get('category_group'),
            'category_type':  parsed.get('category_type'),
            'base_category_key': parsed.get('base_category_key'),
            'merchant':       parsed.get('merchant'),
            'execution_path': parsed.get('execution_path'),
            'confidence':     parsed.get('confidence'),
            'reason':         parsed.get('reason'),
            'flag':           json.dumps(parsed.get('flag')) if parsed.get('flag') else None,
        })
        # Append pipeline steps from pre-processing to execution_path if not already set
        if not result['execution_path'] and signals['execution_path_steps']:
            result['execution_path'] = '->'.join(signals['execution_path_steps']) + '->llm'
    except Exception as e:
        result['parse_error'] = True
        result['reason']      = f'PARSE_ERROR: {e}'
        return result

    # Validate key
    pred_key = result.get('base_category_key', '')
    if pred_key in INVALID_KEY_BLOCKLIST or pred_key not in VALID_KEYS:
        result['invalid_key'] = True

    return result


print('Batch loop functions ready')

In [ ]:
# ---- Single row smoke test -----------------------------------------------
# Run one row before committing to the full batch

smoke_row = df_sample_run.iloc[0].to_dict()
smoke_res = process_row(0, smoke_row)

print('Smoke test result:')
for k, v in smoke_res.items():
    if k not in ('raw_response', 'raw_response_naive'):
        print(f'  {k}: {v}')
print('\nRaw full response:')
print(smoke_res.get('raw_response', ''))
print('\nRaw naive response:')
print(smoke_res.get('raw_response_naive', ''))

In [ ]:
# ---- Full batch run ------------------------------------------------------
# Incremental save after every row. Safe to restart - will append to existing file.

results     = []
total_rows  = len(df_sample_run)
errors      = 0
start_time  = time.time()

# Resume support: if output file exists, skip already-processed row_idx values
processed_idx = set()
if OUTPUT_FILE.exists():
    prev = pd.read_csv(OUTPUT_FILE)
    processed_idx = set(prev['row_idx'].tolist())
    results = prev.to_dict('records')
    print(f'Resuming - {len(processed_idx)} rows already processed')

for i, (_, row) in enumerate(df_sample_run.iterrows()):
    if i in processed_idx:
        continue

    try:
        res = process_row(i, row.to_dict())
    except Exception as e:
        print(f'[{i}] EXCEPTION: {e}')
        errors += 1
        res = {'row_idx': i, 'parse_error': True, 'reason': str(e)}

    results.append(res)

    # Incremental save
    #pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
    pd.DataFrame([res]).to_csv(
        OUTPUT_FILE, 
        mode='a', 
        header=not OUTPUT_FILE.exists() or i == 0,
        index=False
    )

    # Progress
    if (i + 1) % 25 == 0 or i == total_rows - 1:
        elapsed   = time.time() - start_time
        remaining = (elapsed / max(i + 1 - len(processed_idx), 1)) * (total_rows - i - 1)
        print(f'[{i+1}/{total_rows}] errors={errors} elapsed={elapsed:.0f}s est_remaining={remaining:.0f}s')

df_results = pd.DataFrame(results)
print(f'\nBatch complete: {len(df_results)} rows, {errors} errors')
print(f'Output: {OUTPUT_FILE}')

---
## 9. Post-Process and Evaluation

In [ ]:
# Load results if not already in memory
if 'df_results' not in dir():
    df_results = pd.read_csv(OUTPUT_FILE)
    print(f'Loaded {len(df_results)} rows from {OUTPUT_FILE}')

print(f'Total rows: {len(df_results)}')
print(f'Parse errors: {df_results["parse_error"].sum()}')
print(f'Invalid keys: {df_results["invalid_key"].sum()} ({df_results["invalid_key"].mean()*100:.1f}%)')

In [ ]:
# ---- L1 eval: spend_type accuracy ----------------------------------------
# Resolve mixed GT via GT_SPEND_MAP (leans / manual_spend_override)

df_eval = df_results.copy()
df_eval['gt_spend_type'] = df_eval['gt_base_category_key'].map(GT_SPEND_MAP)

# Only evaluate rows where GT spend_type is resolvable
df_l1 = df_eval.dropna(subset=['gt_spend_type', 'spend_type'])
l1_acc = (df_l1['spend_type'] == df_l1['gt_spend_type']).mean()

print(f'L1 spend_type accuracy: {l1_acc*100:.1f}% ({len(df_l1)} evaluable rows)')

In [ ]:
# ---- L2a eval: category_type accuracy (non-spend only) -------------------

# Join GT category_type from taxonomy
tax_ct = tax_df[['base_category_key', 'category_type']].drop_duplicates(subset='base_category_key')
df_eval = df_eval.merge(tax_ct, left_on='gt_base_category_key', right_on='base_category_key',
                        how='left', suffixes=('', '_gt'))
df_eval.rename(columns={'category_type_gt': 'gt_category_type'}, inplace=True)

df_l2a = df_eval[
    (df_eval['gt_spend_type'] == 'non_spend') &
    df_eval['gt_category_type'].notna() &
    df_eval['category_type'].notna()
]
l2a_acc = (df_l2a['category_type'] == df_l2a['gt_category_type']).mean() if len(df_l2a) else 0
print(f'L2a category_type accuracy (non-spend): {l2a_acc*100:.1f}% ({len(df_l2a)} evaluable rows)')

In [ ]:
# ---- L3 eval: exact key accuracy -----------------------------------------

df_l3  = df_eval.dropna(subset=['gt_base_category_key', 'base_category_key'])
l3_acc = (df_l3['base_category_key'] == df_l3['gt_base_category_key']).mean()
print(f'L3 exact key accuracy: {l3_acc*100:.1f}% ({len(df_l3)} rows)')


# L3 by spend type
for st in ['spend', 'non_spend']:
    subset = df_eval[df_eval['gt_spend_type'] == st].copy()
    evaluable = subset[subset['base_category_key'].notna() & subset['gt_base_category_key'].notna()]
    correct = (evaluable['base_category_key'] == evaluable['gt_base_category_key']).sum()
    pct = correct / len(evaluable) * 100 if len(evaluable) > 0 else 0
    print(f'L3 {st}: {pct:.1f}% ({correct}/{len(evaluable)})')


# Top mismatches
mismatches = df_l3[df_l3['base_category_key'] != df_l3['gt_base_category_key']]
mismatch_pairs = mismatches.groupby(['gt_base_category_key', 'base_category_key']).size() \
                            .reset_index(name='count').sort_values('count', ascending=False)
print(f'\nTop 15 mismatch pairs:')
print(mismatch_pairs.head(15).to_string(index=False))

In [ ]:
# ---- Coverage report -----------------------------------------------------

n = len(df_results)

def pct(mask): return f'{mask.sum()} ({mask.mean()*100:.1f}%)'

biller_cov  = df_results['execution_path'].str.startswith('biller').fillna(False)
mcc_cov     = df_results['execution_path'].str.contains('mcc',    na=False)
dt_cov      = df_results['execution_path'].str.contains('dt',     na=False)
merch_cov   = df_results['execution_path'].str.contains('merchant', na=False)
regex_cov   = df_results['execution_path'].str.contains('regex',  na=False)
llm_only    = df_results['execution_path'].fillna('').str.strip() == 'llm'

print('Coverage report:')
print(f'  biller:          {pct(biller_cov)}')
print(f'  mcc:             {pct(mcc_cov)}')
print(f'  dt:              {pct(dt_cov)}')
print(f'  merchant_present:{pct(merch_cov)}')
print(f'  regex:           {pct(regex_cov)}')
print(f'  llm-only:        {pct(llm_only)}')

In [ ]:
# ---- Naive vs pipeline lift ----------------------------------------------

df_lift = df_eval.dropna(subset=['gt_base_category_key', 'base_category_key', 'base_category_key_naive'])

pipeline_l3 = (df_lift['base_category_key']       == df_lift['gt_base_category_key']).mean()
naive_l3    = (df_lift['base_category_key_naive']  == df_lift['gt_base_category_key']).mean()
lift        = pipeline_l3 - naive_l3

print('Naive vs pipeline lift (L3 exact key, same evaluable rows):')
print(f'  Naive (no hints):     {naive_l3*100:.1f}%')
print(f'  Full pipeline:        {pipeline_l3*100:.1f}%')
print(f'  Lift:                 +{lift*100:.1f}pp')

In [ ]:
# ---- Confidence vs accuracy ----------------------------------------------

df_conf = df_eval.dropna(subset=['confidence', 'base_category_key', 'gt_base_category_key']).copy()
df_conf['confidence'] = pd.to_numeric(df_conf['confidence'], errors='coerce')
df_conf['l3_match']   = df_conf['base_category_key'] == df_conf['gt_base_category_key']

conf_summary = df_conf.groupby('confidence').agg(
    count=('l3_match', 'size'),
    accuracy=('l3_match', 'mean')
).reset_index()
conf_summary['accuracy'] = (conf_summary['accuracy'] * 100).round(1)
print('Confidence vs L3 accuracy:')
print(conf_summary.to_string(index=False))

In [ ]:
# ---- Summary print -------------------------------------------------------

print('=' * 50)
print('RUN 3 RESULTS SUMMARY')
print('=' * 50)
print(f'Total rows:         {len(df_results)}')
print(f'Parse errors:       {df_results["parse_error"].sum()} ({df_results["parse_error"].mean()*100:.1f}%)')
print(f'Invalid keys:       {df_results["invalid_key"].sum()} ({df_results["invalid_key"].mean()*100:.1f}%)')
print(f'L1 spend_type:      {l1_acc*100:.1f}% ({len(df_l1)} evaluable)')
print(f'L2a category_type:  {l2a_acc*100:.1f}% (non-spend only, {len(df_l2a)} evaluable)')
print(f'L3 exact key:       {l3_acc*100:.1f}% ({len(df_l3)} evaluable)')
print(f'Naive L3:           {naive_l3*100:.1f}%')
print(f'Pipeline lift:      +{lift*100:.1f}pp')
print(f'Output file:        {OUTPUT_FILE}')


In [ ]:
# What are the remaining invalid keys?
print(df_eval[df_eval['invalid_key'] == True]['base_category_key'].value_counts())

In [ ]:
# Check what's in the taxonomy context being injected
print([k for k in VALID_KEYS if 'RESTAURANT' in k or 'UTIL' in k or 'PARK' in k])

In [ ]:
print('RESTAURANT' in VALID_KEYS)
print('UTILITIES' in VALID_KEYS)
# Check what the actual DB source says
print(cat_keys_df[cat_keys_df['base_category_key'].isin(['RESTAURANT', 'UTILITIES', 'RESTAURANTS__CAFES', 'PARKING'])])

In [ ]:
print('RESTAURANTS__CAFES' in VALID_KEYS)
print('PARKING' in VALID_KEYS)
print(cat_keys_df[cat_keys_df['base_category_key'].isin(['RESTAURANTS__CAFES', 'PARKING'])])

In [ ]:
# How many rows are genuinely invalid vs wrongly flagged?
invalid_rows = df_eval[df_eval['invalid_key'] == True]
print(invalid_rows['base_category_key'].value_counts())

# Are UTILITIES and RESTAURANT in cat_keys_df?
print(f"VALID_KEYS count: {len(VALID_KEYS)}")
print(f"cat_keys_df count: {len(cat_keys_df)}")

In [ ]:
# Find keys in cat_keys_df but missing from VALID_KEYS
cat_keys_set = set(cat_keys_df['base_category_key'])
print("In cat_keys_df but not VALID_KEYS:", cat_keys_set - VALID_KEYS)
print("In VALID_KEYS but not cat_keys_df:", VALID_KEYS - cat_keys_set)